# **Reddit Scraper Notebook for HealthPH+**


# **Dependencies**

In [1]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [2]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
    PLAYWRIGHT_AVAILABLE = True
except ImportError:
    async_playwright = None
    PlaywrightTimeoutError = TimeoutError
    PLAYWRIGHT_AVAILABLE = False


In [3]:
print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## **Reddit**

### Configuration

In [4]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
# Paste any Reddit search URL below.
# The query, sort order, and time filter are parsed automatically from the URL.
# Example filters you can append to the URL:
#   &sort=relevance | new | top | comments
#   &t=all | year | month | week | day | hour


KEYWORD = 'masama pakiramdam'
SEARCH_URL = f"https://www.reddit.com/search/?q={quote_plus(KEYWORD)}&sort=new&t=all"

LIMIT     = 50   # Posts per page (max 100)
MAX_PAGES = 10   # Number of pages to paginate through
REQUEST_DELAY_SECONDS = 2
REQUEST_TIMEOUT_SECONDS = 30
MAX_429_RETRIES = 4
BASE_429_BACKOFF_SECONDS = 30
REDDIT_USER_AGENT = os.getenv(
    "REDDIT_USER_AGENT",
    "HealthPHPlusRedditScraper/0.1 (research; contact: local-notebook)",
)

CWD = Path.cwd()
ROOT_DIR = next((p for p in [CWD, *CWD.parents] if (p / '.git').exists()), CWD)
OUTPUT_FILE = str(ROOT_DIR / "data" / "raw" / "reddit" / "reddit_results.csv")   # Master data output file
# ───────────────────────────────────────────────────────────────────────────────

print(f"Search URL : {SEARCH_URL}")
print(f"Limit      : {LIMIT} posts/page")
print(f"Max pages  : {MAX_PAGES}")
print(f"Page gap   : {REQUEST_DELAY_SECONDS} seconds")
print(f"Output file: {OUTPUT_FILE}")

Search URL : https://www.reddit.com/search/?q=masama+pakiramdam&sort=new&t=all
Limit      : 50 posts/page
Max pages  : 10
Page gap   : 2 seconds
Output file: /Users/gelo-p1/Documents/GitHub/HealthPH-Plus/data/raw/reddit/reddit_results.csv


In [5]:
## 3. Helper Functions
POST_COLUMNS = ["title", "selftext", "created", "ups", "subreddit", "url"]


def empty_posts_df():
    return pd.DataFrame(columns=POST_COLUMNS)


def extract_reddit_search_params(url):
    """
    Extract search query, sort order, and time filter from a Reddit search URL.
    
    Example URL:
        https://www.reddit.com/search/?q=sakit&sort=top&t=month
    
    Returns:
        tuple: (query, sort, time_filter)
    """
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    query = params.get("q", [None])[0]
    sort  = params.get("sort", ["relevance"])[0]
    t     = params.get("t",    ["all"])[0]

    if not query:
        raise ValueError(f"Could not extract a search query from URL: {url}")

    return query, sort, t


def build_search_url(keyword, sort="new", time_filter="all"):
    return (
        "https://www.reddit.com/search/?"
        f"q={quote_plus(keyword)}&sort={quote_plus(sort)}&t={quote_plus(time_filter)}"
    )


def _parse_retry_after_seconds(response, default_seconds):
    retry_after = response.headers.get("Retry-After")
    if not retry_after:
        return default_seconds

    try:
        return max(default_seconds, int(float(retry_after)))
    except (TypeError, ValueError):
        return default_seconds


def reddit_get_with_backoff(base_url, params, headers, timeout_seconds=30, max_retries=4, base_backoff_seconds=30):
    attempt = 0
    while True:
        response = requests.get(base_url, params=params, headers=headers, timeout=timeout_seconds)
        if response.status_code != 429:
            return response

        if attempt >= max_retries:
            print("❌ Reddit kept returning HTTP 429 after all retry attempts.")
            return response

        wait_seconds = _parse_retry_after_seconds(
            response,
            base_backoff_seconds * (2 ** attempt),
        )
        print(
            f"⚠️ HTTP 429 received. Cooling down for {wait_seconds} seconds "
            f"before retry {attempt + 1}/{max_retries}..."
        )
        time.sleep(wait_seconds)
        attempt += 1


def get_reddit_access_token():
    """
    Return an app-only Reddit OAuth token when REDDIT_CLIENT_ID and
    REDDIT_CLIENT_SECRET are set. If they are missing, the scraper falls back
    to the public JSON endpoint and then the browser fallback.
    """
    client_id = os.getenv("REDDIT_CLIENT_ID")
    client_secret = os.getenv("REDDIT_CLIENT_SECRET")
    if not client_id or not client_secret:
        return None

    try:
        response = requests.post(
            "https://www.reddit.com/api/v1/access_token",
            auth=requests.auth.HTTPBasicAuth(client_id, client_secret),
            data={"grant_type": "client_credentials"},
            headers={"User-Agent": REDDIT_USER_AGENT},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as exc:
        print(f"⚠️ Could not request Reddit OAuth token: {exc}")
        return None

    if response.status_code != 200:
        print(f"⚠️ Reddit OAuth token request failed: HTTP {response.status_code}")
        return None

    return response.json().get("access_token")


def _format_created(value):
    if not value:
        return ""

    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value).strftime("%Y-%m-%d %H:%M:%S UTC")

    if isinstance(value, str):
        try:
            parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
            return parsed.strftime("%Y-%m-%d %H:%M:%S UTC")
        except ValueError:
            return value

    return ""


def _normalize_post_url(url):
    if not url:
        return ""
    url = str(url).strip()
    if url.startswith("/"):
        return f"https://reddit.com{url}"
    return url.replace("https://www.reddit.com", "https://reddit.com")


def _post_data_to_row(post_data):
    return {
        "title": post_data.get("title", ""),
        "selftext": post_data.get("selftext", ""),
        "created": _format_created(post_data.get("created_utc")),
        "ups": int(post_data.get("ups") or 0),
        "subreddit": post_data.get("subreddit", ""),
        "url": _normalize_post_url(post_data.get("permalink", "")),
    }


def get_last_reddit_fullname_from_csv(filepath):
    """
    Read the last saved Reddit post URL from CSV and return its fullname (t3_<id>).

    Returns None if the file is missing, empty, or the URL is not parseable.
    """
    if not os.path.isfile(filepath):
        return None

    try:
        df = pd.read_csv(filepath, usecols=["url"])
        if df.empty:
            return None

        last_url = df["url"].dropna().iloc[-1]
        if not isinstance(last_url, str) or not last_url:
            return None

        match = re.search(r"/comments/([a-z0-9]+)/", last_url)
        if not match:
            return None

        return f"t3_{match.group(1)}"
    except Exception as e:
        print(f"⚠️ Could not read last id from {filepath}: {e}")
        return None


async def scrape_reddit_search_browser(url, limit=25, max_pages=3):
    """
    Browser fallback for cases where Reddit blocks unauthenticated search.json.
    Search result cards do not always expose full selftext or exact upvote counts,
    so OAuth remains the preferred path when available.
    """
    query, sort, t = extract_reddit_search_params(url)
    browser_url = build_search_url(query, sort=sort, time_filter=t)
    target_count = max(1, limit) * max(1, max_pages)
    rows = []
    seen_urls = set()

    print("↪ Falling back to Playwright browser search because Reddit blocked JSON access.")
    if not PLAYWRIGHT_AVAILABLE:
        print("❌ Playwright is not installed. Run: pip install playwright && playwright install chromium")
        print("   For the most reliable Reddit access, set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET instead.")
        return empty_posts_df()

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=True)
        context = await browser.new_context(
            user_agent=REDDIT_USER_AGENT,
            locale="en-US",
            viewport={"width": 1280, "height": 900},
        )
        page = await context.new_page()

        try:
            await page.goto(browser_url, wait_until="domcontentloaded", timeout=60_000)
            await page.wait_for_selector(
                "shreddit-post, article, a[href*='/comments/']",
                timeout=20_000,
            )
        except PlaywrightTimeoutError:
            print("❌ Browser fallback timed out waiting for Reddit search results.")
            await browser.close()
            return empty_posts_df()

        for page_index in range(max_pages):
            browser_rows = await page.evaluate(
                r"""
                () => {
                    const cleanUrl = (href) => {
                        if (!href) return "";
                        const absolute = new URL(href, location.origin).toString();
                        return absolute.replace("https://www.reddit.com", "https://reddit.com");
                    };
                    const parseScore = (text) => {
                        if (!text) return 0;
                        const raw = String(text).toLowerCase().replace(/,/g, "").trim();
                        const match = raw.match(/([0-9]+(?:\.[0-9]+)?)(k|m)?/);
                        if (!match) return 0;
                        const value = Number(match[1]);
                        const suffix = match[2];
                        if (suffix === "m") return Math.round(value * 1000000);
                        if (suffix === "k") return Math.round(value * 1000);
                        return Math.round(value);
                    };
                    const cards = Array.from(document.querySelectorAll("shreddit-post, article"));
                    return cards.map((card) => {
                        const link = card.getAttribute("content-href")
                            || card.getAttribute("permalink")
                            || card.querySelector("a[href*='/comments/']")?.getAttribute("href")
                            || "";
                        const title = card.getAttribute("post-title")
                            || card.querySelector("[slot='title'], h1, h2, h3, a[href*='/comments/']")?.textContent
                            || "";
                        const subreddit = card.getAttribute("subreddit-prefixed-name")
                            || card.querySelector("a[href*='/r/']")?.textContent
                            || "";
                        const created = card.getAttribute("created-timestamp")
                            || card.querySelector("time")?.getAttribute("datetime")
                            || "";
                        const score = card.getAttribute("score")
                            || card.querySelector("[id*='vote-arrows'], [aria-label*='upvote'], faceplate-number")?.textContent
                            || "";
                        return {
                            title: title.trim(),
                            selftext: "",
                            created,
                            ups: parseScore(score),
                            subreddit: subreddit.replace(/^r\//, "").trim(),
                            url: cleanUrl(link),
                        };
                    }).filter((row) => row.url.includes("/comments/") && row.title);
                }
                """
            )

            added = 0
            for row in browser_rows:
                normalized_url = _normalize_post_url(row.get("url"))
                if not normalized_url or normalized_url in seen_urls:
                    continue
                seen_urls.add(normalized_url)
                rows.append({
                    "title": row.get("title", ""),
                    "selftext": row.get("selftext", ""),
                    "created": _format_created(row.get("created")),
                    "ups": int(row.get("ups") or 0),
                    "subreddit": row.get("subreddit", ""),
                    "url": normalized_url,
                })
                added += 1
                if len(rows) >= target_count:
                    break

            print(f"✔ Browser pass {page_index + 1}: added {added} posts  (running total: {len(rows)})")
            if len(rows) >= target_count:
                break

            await page.mouse.wheel(0, 3000)
            await page.wait_for_timeout(REQUEST_DELAY_SECONDS * 1000)

        await browser.close()

    return pd.DataFrame(rows, columns=POST_COLUMNS)


async def scrape_reddit_search(url, limit=25, max_pages=3):
    """
    Scrape Reddit search results from a Reddit search URL.

    Args:
        url       : A Reddit search URL.
        limit     : Posts per page (max 100).
        max_pages : Maximum number of pages to paginate through.

    Returns:
        pd.DataFrame with columns: title, selftext, created, ups, subreddit, url
    """
    query, sort, t = extract_reddit_search_params(url)
    print(f"Query: '{query}'  |  Sort: {sort}  |  Time filter: {t}\n")

    access_token = get_reddit_access_token()
    headers = {
        "User-Agent": REDDIT_USER_AGENT,
        "Accept": "application/json",
    }
    if access_token:
        headers["Authorization"] = f"Bearer {access_token}"
        base_url = "https://oauth.reddit.com/search"
        print("Using Reddit OAuth search endpoint.")
    else:
        base_url = "https://www.reddit.com/search.json"
        print("No Reddit OAuth credentials found; using public JSON endpoint first.")

    results = []
    after = None  # pagination cursor (within this run only)

    for page in range(max_pages):
        params = {
            "q": query,
            "limit": limit,
            "sort": sort,
            "t": t,
            "type": "link",   # posts only
        }
        if after:
            params["after"] = after

        response = reddit_get_with_backoff(
            base_url,
            params=params,
            headers=headers,
            timeout_seconds=REQUEST_TIMEOUT_SECONDS,
            max_retries=MAX_429_RETRIES,
            base_backoff_seconds=BASE_429_BACKOFF_SECONDS,
        )

        if response.status_code == 403 and not access_token:
            print("❌ Error: HTTP 403 from public JSON endpoint.")
            return await scrape_reddit_search_browser(url, limit=limit, max_pages=max_pages)

        if response.status_code != 200:
            print(f"❌ Error: HTTP {response.status_code}")
            if response.text:
                print(response.text[:500])
            break

        data = response.json()
        posts = data.get("data", {}).get("children", [])

        if not posts:
            print("ℹ️  No more posts found.")
            break

        for post in posts:
            results.append(_post_data_to_row(post.get("data", {})))

        after = data.get("data", {}).get("after")
        print(f"✔ Page {page + 1}: fetched {len(posts)} posts  (running total: {len(results)})")

        if not after:
            print("ℹ️  Reached last page.")
            break

        time.sleep(REQUEST_DELAY_SECONDS)   # polite delay to avoid rate limiting

    return pd.DataFrame(results, columns=POST_COLUMNS)


def save_results(df, filepath=None):
    """
    Save Reddit posts to CSV without adding duplicates.
    Dedupe key: post URL.
    """
    if filepath is None:
        filepath = OUTPUT_FILE

    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if df.empty:
        print("ℹ️ No rows to save.")
        return

    if "url" not in df.columns:
        raise ValueError("save_results requires a 'url' column for deduplication.")

    file_exists = os.path.isfile(filepath)
    incoming_count = len(df)

    to_save = df.copy()
    to_save["url"] = to_save["url"].astype(str).str.strip()
    to_save = to_save[to_save["url"] != ""]

    # Remove duplicates in this scrape batch
    to_save = to_save.drop_duplicates(subset=["url"], keep="first")

    # Remove rows already present in the output file
    if file_exists and not to_save.empty:
        try:
            existing_urls = set(
                pd.read_csv(filepath, usecols=["url"])["url"]
                .dropna()
                .astype(str)
                .str.strip()
            )
            to_save = to_save[~to_save["url"].isin(existing_urls)]
        except ValueError:
            print("⚠️ Existing file has no 'url' column. Skipping cross-run dedupe.")

    new_rows = len(to_save)
    if new_rows == 0:
        skipped = incoming_count
        print(f"ℹ️ No new posts to append. (Skipped {skipped} duplicates)")
    else:
        to_save.to_csv(
            filepath,
            mode="a",
            index=False,
            encoding="utf-8-sig",
            header=not file_exists,
        )
        skipped = incoming_count - new_rows
        action = "Appended to" if file_exists else "Created"
        print(f"💾 {action} {filepath}  (+{new_rows} posts, skipped {skipped} duplicates)")

    total = pd.read_csv(filepath).shape[0] if os.path.isfile(filepath) else 0
    print(f"📊 Total rows in file: {total}")


print("✅ Functions defined")

✅ Functions defined


### Main Function

In [6]:
df = await scrape_reddit_search(url=SEARCH_URL, limit=LIMIT, max_pages=MAX_PAGES)
print(f"Total posts collected: {len(df)}")


Query: 'masama pakiramdam'  |  Sort: new  |  Time filter: all

No Reddit OAuth credentials found; using public JSON endpoint first.
❌ Error: HTTP 403 from public JSON endpoint.
↪ Falling back to Playwright browser search because Reddit blocked JSON access.
❌ Playwright is not installed. Run: pip install playwright && playwright install chromium
   For the most reliable Reddit access, set REDDIT_CLIENT_ID and REDDIT_CLIENT_SECRET instead.
Total posts collected: 0


### Results Preview 

In [7]:
# First 5 rows
df.head()

,title,selftext,created,ups,subreddit,url


In [8]:
# Upvote distribution
df["ups"].describe()

count       0
unique      0
top       NaN
freq      NaN
Name: ups, dtype: object

In [9]:
# Top 10 posts by upvotes
df.sort_values("ups", ascending=False)[["title", "subreddit", "ups", "created"]].head(10)

,title,subreddit,ups,created


In [10]:
# Post count by subreddit
df["subreddit"].value_counts().head(10)

Series([], Name: count, dtype: int64)

### Save to CSV

In [11]:
save_results(df, OUTPUT_FILE)

ℹ️ No rows to save.


In [12]:
df.sample(10)

ValueError: a must be greater than 0 unless no samples are taken